### 1. Importing libraries that will be used

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import copy
import torch
import glob
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold


### 2. Configuration and Hyperparameter
##### The whole project can be reproduced and changing the hyperparameter from the configuration would provide different outcome

In [ ]:
if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/chest-xray-pneumonia/chest_xray'
else:
    DATA_DIR = 'dataset'

config = {
    'data_dir': DATA_DIR,
    'image_size': 224, # restnet use 224x224 images
    'batch_size': 32,
    'epochs': 10,
    'lr_backbone': 1e-4, # lower learning rate to preserve pretrained weights
    'lr_head': 1e-3, # higher learning rate for the new layers
    'weight_decay': 1e-4,
    'dropout_rate': 0.5,
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'save_path': 'best_resnet_param.pth',
}

torch.manual_seed(config['seed'])   # Reproducibility
np.random.seed(config['seed'])      # Reproducibility

### 3. Data preprocessing and preparation
###### The original source data source has low number of images in validity set, it will create overfitting. Therefore, we will merge train and val set together and then split them with sklearn library. 

In [ ]:
# loading datapath, labeling them and grouping them so stratified sampling can be done later

def dataset_loading_grouping(base_dir):
    filepaths, labels, groups = [],[],[]
    for i in ['train','val']:
        for label, class_name in enumerate(['NORMAL','PNEUMONIA']):
            image_dir = os.path.join(base_dir, i, class_name)
            for ext in ('*.jpeg', '*.jpg', '*.png', '*.JPEG'):
                for image_path in glob.glob(os.path.join(image_dir, ext)): # glob.glob returns a list of file paths
                    filepaths.append(image_path)
                    labels.append(label)
                    filename = os.path.basename(image_path)  # get only the filename from the path
                    group_id = filename.split('_')[0] if 'person' in filename else filename
                    groups.append(group_id)

    return np.array(filepaths), np.array(labels), np.array(groups)


class XrayDataset(Dataset):
    ''''
    This makes sure that images are loaded batch by batch and not all at once
    '''
    def __init__(self, filepaths, labels, transform = None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)
    def __getitem__(self, idx): 
        image_path = self.filepaths[idx]
        image =  Image.open(image_path).convert('RGB') # converting greyscale to rgb for resnet compatibility
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype = torch.float32)
    


### 4. Augmentation
###### To avoid our network to memories every images (as medical images are not too many available), we have to add augmentation to increase the efficiency of our network.

In [ ]:
# each time model load an image, it will supply slight variations of the image.
train_transforms = transforms.Compose(
    [
        transforms.Resize((config['image_size'], config['image_size'])),
        transforms.RandomHorizontalFlip(p = 0.5),
        transforms.RandomRotation(degrees = 10),
        transforms.ColorJitter(brightness = 0.2, contrast = 0.2, saturation = 0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
    ]
)

# this is for validation and test data, 
val_transforms = transforms.Compose(
    [
        transforms.Resize((config['image_size'], config['image_size'])),
        transforms.ToTensor(),
        transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
    ]
)

### 5. Model architecture (ResNet-18)


In [ ]:
def resnet_model(dropout_rate = 0.5):
    model = models.resnet18(weights = models.ResNet18_Weights.IMAGENET1K_V1)
    num_features = model.fc.in_features # get the number of features in the last layer of resnet

    # Replacing last layer with a new layer that is densely connected to a single output neuron (for binary classification) and adding dropout for regularization
    model.fc = torch.nn.Sequential(
        torch.nn.Dropout(dropout_rate),
        torch.nn.Linear(num_features, 1) 
    )

    return model

### 6. Grad-Cam (Gradient weighted class activation)
###### Checking the sanity of the model

In [ ]:
class GradCam:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

    def activation_hook(self, module, input, output):
        self.activations = output.detach()

    def gradient_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate_heatmap(self, input_tensor):
        self.model.eval() #ignores dropout
        output = self.model(input_tensor)
        self.model.zero_grad()
        output.backward() # compute gradients

        weights = torch.mean(self.gradients, dim= (2,3), keepdim = True) # Taking the mean of the gradients across the spatial dimensions (height and width) to get the importance of each channel
        cam = torch.sum(weights * self.activations, dim = 1).squeeze().cpu().numpy() # weighted sum of the activations using the computed weights
        cam = np.maximum(cam,0) # Keeps only the positive values
        cam = (cam - np.min(cam)) / (np.max(cam) - np.min(cam) + 1e-8) # Normalize the heatmap to the range [0,1]
        return cam, torch.sigmoid(output).item() # sigmoid converts logits output to 0-1 probability score

In [ ]:
def loading_data(config):
    filepaths, labels, groups = dataset_loading_grouping(config['data_dir'])
    stratified_split = StratifiedGroupKFold(n_splits = 5)
    train_idx, val_idx = next(stratified_split.split(filepaths, labels, groups))
    
    train_ds = XrayDataset(filepaths[train_idx], labels[train_idx], transform = train_transforms)
    val_ds = XrayDataset(filepaths[val_idx], labels[val_idx], transform = val_transforms)
    
    # loading test data
    test_filepaths,test_labels = [],[]
    for label, class_name in enumerate(['NORMAL','PNEUMONIA']):
        image_dir = os.path.join(config['data_dir'], 'test', class_name)
        for ext in ('*.jpeg', '*.jpg', '*.png', '*.JPEG'):
            for image_path in glob.glob(os.path.join(image_dir, ext)):
                test_filepaths.append(image_path)
                test_labels.append(label)
    
    test_ds = XrayDataset(test_filepaths, test_labels,transform = val_transforms)
    
    
    train_loader = DataLoader(train_ds, batch_size = config['batch_size'], shuffle = True, num_workers = 2)
    val_loader = DataLoader(val_ds, batch_size = config['batch_size'], shuffle = False, num_workers = 2)
    test_loader = DataLoader(test_ds, batch_size = config['batch_size'], shuffle = False, num_workers = 2)
    
    healthy_count = np.sum(labels[train_idx] == 0)
    pneumonia_count = np.sum(labels[train_idx] == 1)
    
    pos_weight = torch.tensor([healthy_count / pneumonia_count], dtype = torch.float32).to(config['device'])
    return train_loader, val_loader, test_loader, test_ds, pos_weight

### 7. Training and evaluationg 

In [ ]:
def train_evaluate(train_loader, train_ds, val_loader, pos_weight, config):
     # weight imbalance

    model = resnet_model(dropout_rate = config['dropout_rate']).to(config['device'])
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight = pos_weight.to(config['device'])) # Binary cross entropy loss and sigmoid

    # Take the backbone parameters followed by fc then pass through optimiser to get updated weight
    backbone_params = [p for name, p in model.named_parameters() if 'fc' not in name] 
    optimiser = torch.optim.AdamW([
        {'params': backbone_params, 'lr': config['lr_backbone']},
        {'params': model.fc.parameters(), 'lr': config['lr_head']},
    ], weight_decay=config['weight_decay'])

    # shrinks the lr slowly following a cosine curve
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimiser, T_max=config['epochs'])

    best_val_auc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for img, label in train_loader:
            img, label = img.to(config['device']), label.to(config['device']).unsqueeze(1)
            optimiser.zero_grad() # wiping old optimizer from previous batch
            output = model(img)
            loss = criterion(output, label)
            loss.backward() # back propagation
            optimiser.step()
            train_loss += loss.item() * img.size(0)
        scheduler.step() # shrinks lr

        # validation
        model.eval()
        val_target, val_pred = [],[]
        with torch.no_grad(): # truning off torch's autograd. No gradient
            for img, label in val_loader:
                img = img.to(config['device'])
                probability = torch.sigmoid(model(img)).cpu().numpy().flatten()
                val_pred.extend(probability)
                val_target.extend(label.numpy())

        val_auc = roc_auc_score(val_target, val_pred)
        print(f"Epoch {epoch+1:02d}/{config['epochs']:02d} | Train Loss: {train_loss/len(train_ds):.4f} | Val ROC-AUC: {val_auc:.4f}")
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    

    model.eval()
    val_targets, val_preds = [],[]
    with torch.no_grad():
        for img, label in val_loader:
            img = img.to(config['device'])
            prob = torch.sigmoid(model(img)).cpu().numpy().flatten()
            val_targets.extend(label.numpy())
            val_preds.extend(prob)

    fpr, tpr, threshold = roc_curve(val_targets, val_preds)
    optimal_idx = np.argmax(tpr[1:] - fpr[1:]) + 1    # test every probability and find the difference of fpr and tpr and return index
    optimal_threshold = threshold[optimal_idx]
    print(f"\nOptimal Threshold (tuned on Val set): {optimal_threshold:.4f}")

    torch.save(
            {'model_state_dict': best_weights,
            'optimal_threshold': optimal_threshold}
            , config['save_path'])
    print(f"Saved best model & threshold ({optimal_threshold:.4f}) to {config['save_path']}")
    return model,optimal_threshold




In [ ]:
def evaluate_test(test_loader, config):

    checkpoint = torch.load(config['save_path'], map_location=config['device'], weights_only=False)
    model = resnet_model(dropout_rate=config['dropout_rate']).to(config['device'])
    model.load_state_dict(checkpoint['model_state_dict'])
    optimal_threshold = checkpoint['optimal_threshold']
    model.eval()

    test_targets, test_preds = [], []
    with torch.no_grad():
        for img, label in test_loader:
            img = img.to(config['device'])
            prob = torch.sigmoid(model(img)).cpu().numpy().flatten()
            test_preds.extend(prob)
            test_targets.extend(label.numpy())

    test_binary_prediction = (np.array(test_preds) >= optimal_threshold).astype(int)

    print(f"\n--- FINAL TEST SET RESULTS (Threshold = {optimal_threshold:.4f}) ---")
    print(f"Test ROC-AUC: {roc_auc_score(test_targets, test_preds):.4f}")
    print(classification_report(test_targets, test_binary_prediction, target_names=["NORMAL", "PNEUMONIA"]))
    print("Confusion Matrix:\n", confusion_matrix(test_targets, test_binary_prediction))

    return model, test_targets, test_preds, test_binary_prediction, optimal_threshold



In [ ]:
def main():
    train_loader, train_ds, val_loader, test_loader, test_ds, pos_weight = loading_data(config)
    train_evaluate(train_loader, train_ds, val_loader, pos_weight, config)
    evaluate_test(test_loader, config)

if __name__ == '__main__':
    main()